###  Total number of encounters by year, quarter, and encounter class


In [0]:

SELECT 
  year,
  quarter,
  encounter_class,

  COUNT(DISTINCT encounter_id) as total,

  ROUND(
    COUNT(DISTINCT encounter_id) * 100.0 / 
    SUM(COUNT(DISTINCT encounter_id)) OVER (PARTITION BY year, quarter),
    2
  ) as percentage

FROM medical_catalog.gold.finalcube 
GROUP BY year, quarter, encounter_class
ORDER BY year, quarter;


### How long Patient stay (Under 24 vs Over 24)

In [0]:

SELECT 
  year,
  month,

  CASE 
    WHEN duration_hours > 24 THEN 'Over 24 hrs'
    ELSE 'Under 24 hrs'
  END as type,

  COUNT(DISTINCT encounter_id) as total,

  ROUND(
    COUNT(DISTINCT encounter_id) * 100.0 / 
    SUM(COUNT(DISTINCT encounter_id)) OVER (PARTITION BY year, month),
    2
  ) as percentage

FROM medical_catalog.gold.finalcube 

GROUP BY year, month, type
ORDER BY year, month;


### Zero Payer Coverage

In [0]:
SELECT 
  year,
  month,
  payer_name,

  COUNT(DISTINCT encounter_id) as total_encounters,

  COUNT(DISTINCT CASE 
      WHEN payer_id IS NULL OR total_claim_cost = 0 
      THEN encounter_id 
  END) as zero_coverage,

  ROUND(
    COUNT(DISTINCT CASE 
        WHEN payer_id IS NULL OR total_claim_cost = 0 
        THEN encounter_id 
    END) * 100.0 
    / COUNT(DISTINCT encounter_id),
    2
  ) as zero_coverage_pct,

  ROUND(
    COUNT(DISTINCT CASE 
        WHEN total_claim_cost > 0 
        THEN encounter_id 
    END) * 100.0 
    / COUNT(DISTINCT encounter_id),
    2
  ) as coverage_pct

FROM medical_catalog.gold.finalcube
GROUP BY year, month, payer_name
ORDER BY year, month;


### Top 10 Procedures in Each Half Year

In [0]:
WITH aggregated AS (
  SELECT
    year,
    half_year,
    procedure_code,
    ROUND(AVG(base_cost),2) AS avg_cost,
    COUNT(*) AS performed_count
  FROM medical_catalog.gold.finalcube 
  GROUP BY year,half_year, procedure_code
)
SELECT *
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY year, half_year
      ORDER BY avg_cost DESC ,performed_count desc
    ) AS row_number
  FROM aggregated
)
WHERE row_number <= 10

### Avg_Total_Claim

In [0]:
SELECT 
  payer_name,
  ROUND(AVG(total_claim_cost),2) as avg_claim_cost

FROM (
  SELECT DISTINCT 
    encounter_id,
    payer_name,
    total_claim_cost
  FROM medical_catalog.gold.finalcube
) t

GROUP BY payer_name
ORDER BY avg_claim_cost DESC;


### Readmission_Rate

In [0]:
WITH base AS (
  SELECT DISTINCT
    encounter_id,
    patient_id,
    start_time,
    end_time,
    year,
    month
  FROM medical_catalog.gold.finalcube
)

, seq AS (
  SELECT 
    *,
    LAG(end_time) OVER (
      PARTITION BY patient_id 
      ORDER BY start_time
    ) as prev_end
  FROM base
)

, flags AS (
  SELECT *,

    CASE 
      WHEN prev_end IS NOT NULL 
           AND start_time >= prev_end
      THEN 1 ELSE 0 
    END as eligible,

    CASE 
      WHEN prev_end IS NOT NULL AND start_time >= prev_end
           AND DATEDIFF(start_time, prev_end) <= 30
      THEN 1 ELSE 0 
    END as readmission

  FROM seq
)

SELECT 
  year,
  month,

  SUM(eligible) as eligible_encounters,
  SUM(readmission) as readmissions,

  ROUND(
    SUM(readmission) * 100.0 / SUM(eligible),
    2
  ) as readmission_rate

FROM flags
GROUP BY year, month
ORDER BY year, month;
